In [0]:

import json
import os
from pyspark.sql.functions import *
src_path = '/Volumes/dbs0808/landing/data/customers'
tracking_fp = f'{src_path}/processed_files.json'

all_files = [f.path for f in dbutils.fs.ls(src_path)]
all_files = [f for f in all_files if not f.endswith('.json')]

if os.path.exists(tracking_fp):
    with open(tracking_fp, 'r') as f:
        processed_files = json.load(f)
else:
    print('I dont find the processed file, i create a new empty one')
    processed_files = []

print(f"Files already processed: {processed_files}")
new_files = [f for f in all_files if f not in processed_files]
print(f"New files will be: {new_files}")

if new_files:
    df = (
        spark.read
        .format('csv')
        .option('header', 'true')
        .option("mergeSchema", "true")
        .load(new_files)
        #.withColumn('ingest_time', current_timestamp())
        )
    df.write.format('delta').mode('append').saveAsTable('bronze.customers_raw')
    processed_files.extend(new_files)

    with open(tracking_fp, 'w') as f:
        json.dump(processed_files, f)
    print(f"updated processed_files { processed_files}")
else:
    print("No new files processed")